# 构造微调训练数据集

借助 ChatGPT 和 GPT API 我们可以实现自动化批量构造训练数据集。

下面我们以中国古典哲学数据集为例，展示了自动构造训练集的主要流程：

- 使用 LangChain 构造训练数据样例
    - 基于 ChatGPT 设计 `System Role` 提示词
    - 使用 `OpenAI GPT-4o-mini` 生成基础数据
    - 解析 OpenAI GPT 生成的训练数据
    - 持久化存储`dataset.csv`训练数据集文件
    - 使用 ChatGPT 实现训练数据多样化
- 自动化批量生成训练数据集
    - 整理收集原始数据`raw_data.txt`
    - 自动解析原始数据样例 `raw_data_content[]`
    - 设计 `gen_data` 训练数据生成器函数
    - 设计训练数据生成流水线

最佳实践参考：

- 使用 GPT-4o-mini 生成基础数据：https://platform.openai.com/playground/p/2c7XNPgo6Y2iDxILiWfD3iPu?model=gpt-4o-mini&mode=chat
- 使用 ChatGPT 生成数据处理代码和相关文本整理：https://chat.openai.com/share/cdfd2d1d-a75e-4cee-be49-539c010ca1b1
- GPT API 价格: https://openai.com/pricing

## 使用 OpenAI SDk 构造训练数据

In [14]:
import os
from pathlib import Path

from dotenv import load_dotenv
import httpx
from openai import OpenAI

# 从当前工作目录一路向上查找 .env（不限层级）
_root = Path.cwd().resolve()
_env_path_used = None
for _dir in [_root, *_root.parents]:
    _env = _dir / ".env"
    if _env.is_file():
        # utf-8-sig：避免 Windows 记事本「UTF-8 带 BOM」导致读不到 OPENAI_API_KEY
        load_dotenv(_env, override=True, encoding="utf-8-sig")
        _env_path_used = _env
        break
if _env_path_used is None:
    load_dotenv(override=True, encoding="utf-8-sig")

api_key = (os.environ.get("OPENAI_API_KEY") or "").strip()
base_url = (os.environ.get("OPENAI_BASE_URL") or "https://api.openai.com/v1").strip()

def _mask_key(k: str) -> str:
    if not k:
        return "(空)"
    if len(k) <= 12:
        return k[:4] + "..."
    return k[:7] + "..." + k[-4:]

print("[OpenAI 环境诊断]")
print("  cwd:", _root)
print("  已加载 .env:", _env_path_used if _env_path_used else "(未找到文件，仅用系统环境变量 / 空)")
print("  OPENAI_BASE_URL:", base_url)
print("  OPENAI_API_KEY:", _mask_key(api_key), "| 长度:", len(api_key))

if not api_key:
    raise ValueError(
        "未读到 OPENAI_API_KEY。请确认：1) 项目根目录存在 .env；2) 内有 OPENAI_API_KEY=...；"
        "3) 若上面「已加载 .env」为未找到，请把 notebook 的工作目录改到含 .env 的目录或在根目录启动 Jupyter。"
    )

# 显式传入 httpx.Client，避免旧版 openai 内部创建带 proxies= 的客户端（与 httpx 0.28+ 不兼容）
_http_client = httpx.Client(
    base_url=base_url,
    timeout=httpx.Timeout(120.0, connect=15.0),
    limits=httpx.Limits(max_connections=1000, max_keepalive_connections=100),
    follow_redirects=True,
)
client = OpenAI(api_key=api_key, base_url=base_url, http_client=_http_client)
print("  OpenAI 客户端已创建（若下面请求仍失败，多为网络/鉴权/模型名问题，见下一格诊断）")


[OpenAI 环境诊断]
  cwd: /Users/xizhi/Code/LLM-quickstart/chatglm
  已加载 .env: /Users/xizhi/Code/LLM-quickstart/.env
  OPENAI_BASE_URL: https://api.apiyi.com/v1
  OPENAI_API_KEY: sk-JxmH...F5B4 | 长度: 51
  OpenAI 客户端已创建（若下面请求仍失败，多为网络/鉴权/模型名问题，见下一格诊断）


In [15]:
# 可选诊断：发一条最小请求，区分「配置已读到但 API 报错」与别的问题
try:
    _probe = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "ping"}],
        max_tokens=5,
    )
    print("[API 探测] 成功，回复:", (_probe.choices[0].message.content or "")[:100])
except Exception as e:
    print("[API 探测] 失败:", type(e).__name__, e)
    body = getattr(e, "body", None)
    if body is not None:
        print("  body:", body)
    resp = getattr(e, "response", None)
    if resp is not None:
        print("  HTTP:", getattr(resp, "status_code", None))
        try:
            print("  text:", (resp.text or "")[:800])
        except Exception:
            pass

[API 探测] 成功，回复: Pong! How can


In [16]:
response = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
    {
      "role": "system",
      "content": "你是中国古典哲学大师，尤其擅长周易的哲学解读。\n\n接下来，你收到的都是关于周易卦象的解释，你需要整理润色，并生成用于大模型训练的内容和格式。\n\n示例输入：\n\n师卦，此卦是异卦相叠，下卦为坎，上卦为坤。“师”指军队。坎为水、为险；坤为地、为顺，喻寓兵于农。兵凶战危，用兵乃圣人不得已而为之，但它可以顺利无阻碍地解决矛盾，因为顺乎形势，师出有名，故能化凶为吉。占得此卦，对于军事上率师出征非常有利，必无灾祸。师卦是天马出群之卦，以寡伏众之象。\n师卦位于讼卦之后，《序卦》之中这样解释道：“讼必有众起，故受之以师。师者，众也。”争讼的人越来越多，以致形成了军队。\n\n期待结果：\n\ncontent:\"师卦\"\nsummary:\"在周易中，师卦是一个极具深意的卦象，它由两个异卦相叠组成：下卦坎（水）和上卦坤（地）。这一卦象代表“师”，即军队，寓意着兵力和农力的结合。在这里，坎卦象征着水和险难，而坤卦象征着地和顺从，暗示着通过将军事力量安置于民间，可以在必要时顺利调动。\n\n师卦的核心哲学是：虽然兵力代表着危险和战争，但其使用应当是圣人不得已而为之的最后手段。在正确的情况下，军事力量可以顺应形势，将危险转化为吉祥。因此，在军事策略上，此卦象征着出征将会顺利，无灾祸。\n\n师卦紧随讼卦（争讼卦），在《序卦》中解释为“讼必有众起，故受之以师”。这意味着争端激化至众多人群的参与，形成了类似军队的集体力量。\""
    },
    {
      "role": "user",
      "content": "蒙卦是教育启蒙的智慧，艮为山，坎为泉，山下出泉。泉水始流出山，则必将渐汇成江河,正如蒙稚渐启，又山下有险，因为有险停止不前，所以蒙昧不明。事物发展的初期阶段，必然蒙昧，所以教育是当务之急，养学生纯正无邪的品质，是治蒙之道。\n蒙卦，这个卦是异卦相叠，下卦为坎，上卦为艮。艮是山的形象，喻止；坎是水的形象，喻险。卦形为山下有险，仍不停止前进，是为蒙昧，故称蒙卦。但因把握时机，行动切合时宜;因此，具有启蒙和通达的卦象。\n《蒙》卦是《屯》卦这个始生卦之后的第二卦。《序卦》中说：“物生必蒙，故受之以蒙。蒙者，蒙也，特之稚也。”物之幼稚阶段，有如蒙昧未开的状态，在人则是指童蒙。\n《象》中这样解释蒙卦：山下出泉，蒙；君子以果行育德。"
    },
    {
      "role": "assistant",
      "content": "content:\"蒙卦\"\nsummary:\"在周易中，蒙卦象征着教育启蒙和智慧的初期阶段，由两个异卦相叠组成：下卦坎（水）和上卦艮（山）。坎象征水，艮象征山，代表了山下有水源，水流出山。这预示着在事物发展的起初阶段，人们会陷入蒙昧状态，因此教育成为当务之急。只有培养纯正无邪的品质，才能走上通达之路，如同泉水汇成江河。\n\n蒙卦的核心哲学是：事物发展的初期阶段必然处于蒙昧状态，但如果能把握时机，行动切合时宜，就能启蒙通达。蒙卦紧随始生卦《屯》之后，《序卦》中解释道：“物生必蒙，故受之以蒙。蒙者，蒙也，特之稚也。”这说明物在幼稚阶段都会经历蒙昧状态，找它蒙即是稚嫩。\n\n《象》中解释说：山下出泉，蒙；君子以果行育德，意味着像泉水从山下流出一样，启蒙教育有助于培育美德。\""
    },
    {
      "role": "user",
      "content": "需卦，本卦是异卦相叠，上卦为坎，下卦为乾。乾卦象征天空辽阔，坎卦象征乌云密布。雨水之于传统农业社会而言，是最大的需求，因此用这个意象来表现。古人引申为雨将下，万物皆待，所以叫需。\n需，古体字结构是“雨上天下”，从雨而声。如苍天下雨，滋养万物及人。从人的角度看，天下雨，则不易出行耕作，因此在家喝酒吃肉、静待天时。所以，这一卦的基本象征就是等待、不进，还有饮食之义。\n \n需卦位于蒙卦之后，《序卦》之中这样解释道：“物稚不可不养也，故受之以需。需者，饮食之道也。”有所需要，也是有所等待。\n《象》曰：云上于天，需;君子以饮食宴乐。\n《象》中这段话的意思是说：需卦的卦象是乾(天)下坎(水)上，为水在天上之表象。水汽聚集天上成为云层，密云満天，但还没有下雨，需要等待;君子在这个时候需要吃喝，饮酒作乐，即在等待的时候积蓄力量。\n需卦象征等待，启示人们守正待机的道理，属于中上卦。《象》这样评断此卦：明珠土埋日久深，无光无亮到如今，忽然大风吹土去，自然显露有重新。\n"
    },
    {
      "role": "assistant",
      "content": "content:\"需卦\"\nsummary:\"在周易中，需卦是一个富有深意的卦象，由两个异卦相叠组成：下卦乾（天）和上卦坎（水）。乾象征广阔的天空，而坎象征乌云密布，暗示着迫切需要雨水的意象。古人将雨水视为万物生长的必需，因此命名为“需”。需卦的基本象征是等待和饮食之道。\n\n需卦位于蒙卦之后，《序卦》中解释为“物稚不可不养也，故受之以需。需者，饮食之道也。”说明物在幼稚状态需要得到滋养，因此受到需卦的启示。《象》中描述：云上于天，需；君子以饮食宴乐。意味着天空乌云密布，需要等待雨水，而君子则在等待的过程中积蓄力量，享受饮食和欢乐。\n\n需卦的核心哲学是：等待，启示着守正待机的道理。它属于中上卦，象征着埋藏已久的明珠，经过大风吹去尘埃后，重新显露光芒。\""
    }
  ],
  temperature=1,
  max_tokens=4095,
  top_p=1,
  frequency_penalty=0,
  presence_penalty=0
)

---

## 使用 LangChain 构造训练数据

In [20]:
import os
from pathlib import Path

from dotenv import load_dotenv
import httpx
# 必须用 langchain_openai：langchain_community 旧版把同一 httpx.Client 传给 AsyncOpenAI 会报错
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)

_root = Path.cwd().resolve()
for _dir in [_root, *_root.parents]:
    _env = _dir / ".env"
    if _env.is_file():
        load_dotenv(_env, override=True, encoding="utf-8-sig")
        break
else:
    load_dotenv(override=True, encoding="utf-8-sig")

_key = (os.environ.get("OPENAI_API_KEY") or "").strip()
_base = (os.environ.get("OPENAI_BASE_URL") or "https://api.openai.com/v1").strip()
# 显式传入 httpx 客户端，避免旧版 openai 与 httpx 0.28+ 的 proxies 冲突
_lc_timeout = httpx.Timeout(120.0, connect=15.0)
_lc_limits = httpx.Limits(max_connections=1000, max_keepalive_connections=100)
_http_lc = httpx.Client(
    base_url=_base, timeout=_lc_timeout, limits=_lc_limits, follow_redirects=True
)
_http_lc_async = httpx.AsyncClient(
    base_url=_base, timeout=_lc_timeout, limits=_lc_limits, follow_redirects=True
)
chat = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=1,
    max_tokens=4095,
    openai_api_key=_key,
    openai_api_base=_base,
    http_client=_http_lc,
    http_async_client=_http_lc_async,
)

In [21]:
system_content = """
你是中国古典哲学大师，尤其擅长周易的哲学解读。

接下来，你收到的都是关于周易卦象的解释，你需要整理润色，并生成用于大模型训练的内容和格式。

示例输入：

师卦，此卦是异卦相叠，下卦为坎，上卦为坤。“师”指军队。坎为水、为险；坤为地、为顺，喻寓兵于农。兵凶战危，用兵乃圣人不得已而为之，但它可以顺利无阻碍地解决矛盾，因为顺乎形势，师出有名，故能化凶为吉。占得此卦，对于军事上率师出征非常有利，必无灾祸。师卦是天马出群之卦，以寡伏众之象。
师卦位于讼卦之后，《序卦》之中这样解释道：“讼必有众起，故受之以师。师者，众也。”争讼的人越来越多，以致形成了军队。

期待结果：

content:"师卦"
summary:"在周易中，师卦是一个极具深意的卦象，它由两个异卦相叠组成：下卦坎（水）和上卦坤（地）。这一卦象代表“师”，即军队，寓意着兵力和农力的结合。在这里，坎卦象征着水和险难，而坤卦象征着地和顺从，暗示着通过将军事力量安置于民间，可以在必要时顺利调动。

师卦的核心哲学是：虽然兵力代表着危险和战争，但其使用应当是圣人不得已而为之的最后手段。在正确的情况下，军事力量可以顺应形势，将危险转化为吉祥。因此，在军事策略上，此卦象征着出征将会顺利，无灾祸。

师卦紧随讼卦（争讼卦），在《序卦》中解释为“讼必有众起，故受之以师”。这意味着争端激化至众多人群的参与，形成了类似军队的集体力量。"
"""


In [22]:
# 原始数据
raw_content = "蒙卦是教育启蒙的智慧，艮为山，坎为泉，山下出泉。泉水始流出山，则必将渐汇成江河,正如蒙稚渐启，又山下有险，因为有险停止不前，所以蒙昧不明。事物发展的初期阶段，必然蒙昧，所以教育是当务之急，养学生纯正无邪的品质，是治蒙之道。\n蒙卦，这个卦是异卦相叠，下卦为坎，上卦为艮。艮是山的形象，喻止；坎是水的形象，喻险。卦形为山下有险，仍不停止前进，是为蒙昧，故称蒙卦。但因把握时机，行动切合时宜;因此，具有启蒙和通达的卦象。\n《蒙》卦是《屯》卦这个始生卦之后的第二卦。《序卦》中说：“物生必蒙，故受之以蒙。蒙者，蒙也，特之稚也。”物之幼稚阶段，有如蒙昧未开的状态，在人则是指童蒙。\n《象》中这样解释蒙卦：山下出泉，蒙；君子以果行育德。"

In [23]:
messages = [
    SystemMessage(
        content=system_content
    ),
    HumanMessage(
        content=raw_content
    ),
]

In [24]:
ai_message = chat(messages)

/Users/xizhi/Code/LLM-quickstart/.venv/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


### 解析 OpenAI GPT 生成的训练数据

In [25]:
ai_message.content

'content:"蒙卦"\nsummary:"在周易中，蒙卦象征着教育启蒙的智慧，由下卦坎（水）和上卦艮（山）异卦相叠而成。艮卦代表山，象征着静止与停滞，而坎卦则象征水，具备险阻的特性。蒙卦的形象意味着山下出泉，暗示着尽管面临险阻，仍然需要不断前行，象征着从蒙昧走向启明的过程。\n\n蒙卦体现的核心哲学是：在事物发展的初期阶段，往往会经历一种蒙昧的状态，因此教育的启蒙尤为重要，必要时需要引导学生培养纯正无邪的品德，这是解开蒙昧之道的关键。行动与时机的把握能够推动启蒙和理解的过程，使得从蒙昧走向明晓成为可能。\n\n值得注意的是，蒙卦是继屯卦（始生卦）之后的第二卦。《序卦》中提到：\'物生必蒙，故受之以蒙。蒙者，蒙也，特之稚也。\'这句话强调了事物幼稚阶段的特点，如同处于蒙昧中的童蒙。\n\n《象》亦指出：\'山下出泉，蒙；君子以果行育德。\'这一教义再次强调了启蒙教育的重要性以及通过果断的行动来培养德行的必要性。"'

In [26]:
text = ai_message.content

# 分割字符串来找到content和summary的位置
content_start = text.find('content:"') + len('content:"')
content_end = text.find('"\nsummary:')
summary_start = text.find('summary:"') + len('summary:"')
summary_end = text.rfind('"')

# 提取并存储content和summary
content = text[content_start:content_end].strip()
summary = text[summary_start:summary_end].strip()

print("Content:", content)
print("Summary:", summary)


Content: 蒙卦
Summary: 在周易中，蒙卦象征着教育启蒙的智慧，由下卦坎（水）和上卦艮（山）异卦相叠而成。艮卦代表山，象征着静止与停滞，而坎卦则象征水，具备险阻的特性。蒙卦的形象意味着山下出泉，暗示着尽管面临险阻，仍然需要不断前行，象征着从蒙昧走向启明的过程。

蒙卦体现的核心哲学是：在事物发展的初期阶段，往往会经历一种蒙昧的状态，因此教育的启蒙尤为重要，必要时需要引导学生培养纯正无邪的品德，这是解开蒙昧之道的关键。行动与时机的把握能够推动启蒙和理解的过程，使得从蒙昧走向明晓成为可能。

值得注意的是，蒙卦是继屯卦（始生卦）之后的第二卦。《序卦》中提到：'物生必蒙，故受之以蒙。蒙者，蒙也，特之稚也。'这句话强调了事物幼稚阶段的特点，如同处于蒙昧中的童蒙。

《象》亦指出：'山下出泉，蒙；君子以果行育德。'这一教义再次强调了启蒙教育的重要性以及通过果断的行动来培养德行的必要性。


### 持久化存储训练数据集文件

In [27]:
import csv

# 如果没有GPT API，可以使用预定义的变量
# content = "蒙卦"
# summary = "在周易中，师卦是一个极具深意的卦象，它由两个异卦相叠组成：下卦坎（水）和上卦坤（地）。这一卦象代表“师”，即军队，寓意着兵力和农力的结合。在这里，坎卦象征着水和险难，而坤卦象征着地和顺从，暗示着通过将军事力量安置于民间，可以在必要时顺利调动。师卦的核心哲学是：虽然兵力代表着危险和战争，但其使用应当是圣人不得已而为之的最后手段。在正确的情况下，军事力量可以顺应形势，将危险转化为吉祥。因此，在军事策略上，此卦象征着出征将会顺利，无灾祸。师卦紧随讼卦（争讼卦），在《序卦》中解释为“讼必有众起，故受之以师”。这意味着争端激化至众多人群的参与，形成了类似军队的集体力量。"

# 新建CSV文件并写入数据
with open('test_dataset.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    # 写入标题行
    writer.writerow(['content', 'summary'])
    # 写入数据行
    writer.writerow([content, summary])


### 数据增强：构造多样化的提问方式

In [28]:
def generate_question_summary_pairs(content, summary):
    """
    生成20对提问和总结的配对。

    :param content: 内容（例如：“蒙卦”）。
    :param summary: 内容的总结。
    :return: 包含20对提问和总结的列表。
    """
    # 20种提问模板
    question_templates = [
        "{}代表什么？",
        "周易中的{}含义是什么？",
        "请解释一下{}。",
        "{}在周易中是什么象征？",
        "周易{}的深层含义是什么？",
        "{}和教育启蒙有什么联系？",
        "周易的{}讲述了什么？",
        "{}是怎样的一个卦象？",
        "{}在周易中怎样表达教育的概念？",
        "{}的基本意义是什么？",
        "周易中{}的解释是什么？",
        "{}在周易中代表了哪些方面？",
        "{}涉及哪些哲学思想？",
        "周易中{}的象征意义是什么？",
        "{}的主要讲述内容是什么？",
        "周易{}的核心思想是什么？",
        "{}和启蒙教育之间有何联系？",
        "在周易中，{}象征着什么？",
        "请描述{}的含义。",
        "{}在周易哲学中扮演什么角色？"
    ]

    # 使用content填充提问模板
    questions = [template.format(content) for template in question_templates]

    # 创建提问和总结的配对
    question_summary_pairs = [(question, summary) for question in questions]

    return question_summary_pairs

In [29]:
import csv

# 如果没有GPT API，可以使用预定义的变量
# content = "蒙卦"
# summary = "在周易中，师卦是一个极具深意的卦象，它由两个异卦相叠组成：下卦坎（水）和上卦坤（地）。这一卦象代表“师”，即军队，寓意着兵力和农力的结合。在这里，坎卦象征着水和险难，而坤卦象征着地和顺从，暗示着通过将军事力量安置于民间，可以在必要时顺利调动。师卦的核心哲学是：虽然兵力代表着危险和战争，但其使用应当是圣人不得已而为之的最后手段。在正确的情况下，军事力量可以顺应形势，将危险转化为吉祥。因此，在军事策略上，此卦象征着出征将会顺利，无灾祸。师卦紧随讼卦（争讼卦），在《序卦》中解释为“讼必有众起，故受之以师”。这意味着争端激化至众多人群的参与，形成了类似军队的集体力量。"
pairs = generate_question_summary_pairs(content, summary)

# 将结果写入CSV文件
with open('test_dataset.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['content', 'summary'])
    for pair in pairs:
        writer.writerow(pair)


## 自动化批量生成训练数据流水线

原始数据来源：https://www.zhouyi.cc/zhouyi/yijing64/4103.html

In [30]:
# 初始化一个空列表用于存储原始内容数据
raw_content_data = []

# 读取文件并分割数据样例
with open('data/raw_data.txt', 'r', encoding='utf-8') as file:
    content = file.read()
    # 使用连续的换行符('\n\n')作为分隔符来分割文本
    data_samples = content.split('\n\n')

    # 遍历分割后的数据样例并添加到列表中
    for sample in data_samples:
        # 移除每个样例中的额外空白字符（如果有的话）
        cleaned_sample = sample.strip()
        # 仅添加非空样例
        if cleaned_sample:
            raw_content_data.append(cleaned_sample)

In [31]:
# 输出结果以验证
for i, sample in enumerate(raw_content_data[:5]):  # 打印前5个样例以检查
    print(f"样例 {i+1}:")
    print(sample)
    print("------")


样例 1:
蒙卦原文
蒙。亨。匪我求童蒙，童蒙求我。初筮告，再三渎，渎则不告。利贞。
象曰：山下出泉，蒙。君子以果行育德。
白话文解释
蒙卦：通泰。不是我有求于幼稚愚昧的人，而是幼稚愚昧的人有求于我。第一次占筮，神灵告诉了他。轻慢不敬的再三占筮，轻慢不敬的占筮，神灵就不会告诉他。但还是吉利的卜问。
《象辞》说：上卦为艮，象征山；下卦为坎，象征泉。山下有泉，泉水喷涌而出，这是蒙卦的卦象。君子观此卦象，取法于一往无前的山泉，从而以果敢坚毅的行动来培养自身的品德。
《断易天机》解
蒙卦艮上坎下，为离宫四世卦。蒙即蒙昧，主回还往复，疑惑不前，多忧愁过失，乃是凶卦。
北宋易学家邵雍解
智慧未开，蒙昧闭塞；犹豫不决，缺乏果断。
得此卦者，智慧犹如童蒙，不辨是非，迷失方向；若能顺贤师良友之教，启其聪明则亨通。
台湾国学大儒傅佩荣解
时运：蓄积德行，出而用世。
财运：矿山生意，果决则吉。
家宅：君子居吉；婚姻之始。
身体：驱去邪热，可保平安。
传统解卦
这个卦是异卦（下坎上艮）相叠，艮是山的形象，喻止；坎是水的形象，喻险。卦形为山下有险，仍不停止前进，是为蒙昧，故称蒙卦。但因把握时机，行动切合时宜，因此，具有启蒙和通达的卦象。
大象：蒙者，昏而无所见也，故宜「启蒙」。
运势：初时迷惑不知方向，须忍耐待机而动，凡事多听取别人意见，则运可通。
事业：事业开始，混乱无序，危机四伏，以勇敢坚毅的行动可以扭转局面。然而必须接受严格教育，培养这种奋发图强的精神。务必脚踏实地，最忌好高骛远，否则会陷入孤立无援的境地。
经商：务必小心谨慎，不得急功近利，尤其应树立高尚的商业道德，以良好的信誉提高竞争力而取胜。
求名：必须接受良好的基础教育，陶冶情操。且动机纯正，可以达到目的。
婚恋：注意考察对方品德，不可以金钱为诱铒。夫妻需相互宽容、理解。
决策：有时会陷入迷惘困顿的境地，加上胆小、不果断，往往误事。如能接受长辈的教诲，甚至严酷的考验，抛弃疑惧的心理，等待适当时机，必然一帆风顺。
------
样例 2:
屯卦原文
屯。元，亨，利，贞。勿用，有攸往，利建侯。
象曰：云，雷，屯；君子以经纶。
白话文解释
屯卦。大吉大利，吉利的占卜。不利于出门。有利于建国封侯。
《象辞》说：屯的上卦为坎，坎为云，下卦为震，震为雷。云行于上，雷动于下，是屯卦的卦象。君子观此卦象，取法于云雷，用云的恩泽，雷的威严来治理国

### 将以上的所有模块，整合到一起，自动化生成数据

In [33]:
import os
from pathlib import Path

from dotenv import load_dotenv
import httpx
# 必须用 langchain_openai：langchain_community 旧版把同一 httpx.Client 传给 AsyncOpenAI 会报错
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts.chat import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    SystemMessagePromptTemplate,
)

_root = Path.cwd().resolve()
for _dir in [_root, *_root.parents]:
    _env = _dir / ".env"
    if _env.is_file():
        load_dotenv(_env, override=True, encoding="utf-8-sig")
        break
else:
    load_dotenv(override=True, encoding="utf-8-sig")

# 初始化LangChain的GPT-4o-mini调用
_key = (os.environ.get("OPENAI_API_KEY") or "").strip()
_base = (os.environ.get("OPENAI_BASE_URL") or "https://api.openai.com/v1").strip()
_lc_timeout = httpx.Timeout(120.0, connect=15.0)
_lc_limits = httpx.Limits(max_connections=1000, max_keepalive_connections=100)
_http_lc = httpx.Client(
    base_url=_base, timeout=_lc_timeout, limits=_lc_limits, follow_redirects=True
)
_http_lc_async = httpx.AsyncClient(
    base_url=_base, timeout=_lc_timeout, limits=_lc_limits, follow_redirects=True
)
chat = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=1,
    max_tokens=4095,
    openai_api_key=_key,
    openai_api_base=_base,
    http_client=_http_lc,
    http_async_client=_http_lc_async,
)

def gen_data(raw_content):
    """
    使用LangChain GPT-4o-mini调用处理单个数据样例。

    :param raw_content: 原始数据样例。
    :return: GPT-4o-mini模型生成的内容。
    """
    # 系统消息定义背景和任务
    system_message = SystemMessage(
        content="""
        你是中国古典哲学大师，尤其擅长周易的哲学解读。

        接下来，你收到的都是关于周易卦象的解释，你需要整理润色，并生成用于大模型训练的内容和格式。

        示例输入：

        师卦，此卦是异卦相叠，下卦为坎，上卦为坤。“师”指军队。坎为水、为险；坤为地、为顺，喻寓兵于农。兵凶战危，用兵乃圣人不得已而为之，但它可以顺利无阻碍地解决矛盾，因为顺乎形势，师出有名，故能化凶为吉。占得此卦，对于军事上率师出征非常有利，必无灾祸。师卦是天马出群之卦，以寡伏众之象。
        师卦位于讼卦之后，《序卦》之中这样解释道：“讼必有众起，故受之以师。师者，众也。”争讼的人越来越多，以致形成了军队。

        期待结果：

        content:"师卦"
        summary:"在周易中，师卦是一个极具深意的卦象，它由两个异卦相叠组成：下卦坎（水）和上卦坤（地）。这一卦象代表“师”，即军队，寓意着兵力和农力的结合。在这里，坎卦象征着水和险难，而坤卦象征着地和顺从，暗示着通过将军事力量安置于民间，可以在必要时顺利调动。

        师卦的核心哲学是：虽然兵力代表着危险和战争，但其使用应当是圣人不得已而为之的最后手段。在正确的情况下，军事力量可以顺应形势，将危险转化为吉祥。因此，在军事策略上，此卦象征着出征将会顺利，无灾祸。

        师卦紧随讼卦（争讼卦），在《序卦》中解释为“讼必有众起，故受之以师”。这意味着争端激化至众多人群的参与，形成了类似军队的集体力量。"
        """
    )

    # 人类消息包含原始数据样例
    human_message = HumanMessage(
        content=raw_content
    )

    # 构建消息列表并进行模型调用
    messages = [system_message, human_message]
    ai_message = chat(messages)

    return ai_message.content

In [34]:
# 示例调用（使用 raw_data.txt 中解析的数据样例）
generated_content = gen_data(raw_content_data[0])
print(generated_content)

content:"蒙卦"
summary:"在周易中，蒙卦象征着启示与启蒙，其卦象由上卦艮（山）和下卦坎（泉）组成，描绘了山下涌出泉水的画面。这一卦旨在提醒君子在面对蒙昧与迷惘时，要以果敢坚定的行动态度来培养自身的德行。

蒙卦的核心思想是：在初始的迷惑时期，智慧未能完全显露，仿佛儿童般无知。在此阶段，应该以谦逊的态度，待人以诚，倾听他人的意见，避免盲目行动。过于轻慢或不敬的占筮，神灵将不予回应，而正确恰当的提问则会带来吉利的指引。

本卦的象辞指出，通过观察山下泉水的涌现，君子应以此为借鉴，培养坚毅果敢的品德，积极推动自身向前发展。尽管初期会面临因迷惑而带来的忧愁和不安，但如果能够接受良师的教导，顺应时势，将能迎来转机。

此外，蒙卦还反映出事业和人生的多样性。在事业的起步阶段，可能会面临混乱与危机亟待应对，只有通过正确的教育和坚毅的行动才能扭转局势。在商业上需谨慎行事，树立高尚的道德规范以及良好的信用，以增强竞争力。

总体而言，蒙卦告诫我们：在面对不确定性时应坚持学习与成长，切勿急于求成，同时在与人交往时，要关注他人的品德和真诚动机，方能在人生道路上通达顺利。"


In [35]:
def dataset_parser(ai_message_content):
    """
    解析由gen_data函数生成的ai_message.content，提取content和summary。

    :param ai_message_content: gen_data函数返回的文本。
    :return: 提取的content和summary。
    """
    # 分割字符串来找到content和summary的位置
    content_start = ai_message_content.find('content:"') + len('content:"')
    content_end = ai_message_content.find('"\nsummary:')
    summary_start = ai_message_content.find('summary:"') + len('summary:"')
    summary_end = ai_message_content.rfind('"')

    # 提取并存储content和summary
    content = ai_message_content[content_start:content_end].strip()
    summary = ai_message_content[summary_start:summary_end].strip()

    return content, summary


In [36]:
# 示例调用（使用假设的gen_data函数返回的文本）
content, summary = dataset_parser(generated_content)
print("Content:", content)
print("Summary:", summary)

Content: 蒙卦
Summary: 在周易中，蒙卦象征着启示与启蒙，其卦象由上卦艮（山）和下卦坎（泉）组成，描绘了山下涌出泉水的画面。这一卦旨在提醒君子在面对蒙昧与迷惘时，要以果敢坚定的行动态度来培养自身的德行。

蒙卦的核心思想是：在初始的迷惑时期，智慧未能完全显露，仿佛儿童般无知。在此阶段，应该以谦逊的态度，待人以诚，倾听他人的意见，避免盲目行动。过于轻慢或不敬的占筮，神灵将不予回应，而正确恰当的提问则会带来吉利的指引。

本卦的象辞指出，通过观察山下泉水的涌现，君子应以此为借鉴，培养坚毅果敢的品德，积极推动自身向前发展。尽管初期会面临因迷惑而带来的忧愁和不安，但如果能够接受良师的教导，顺应时势，将能迎来转机。

此外，蒙卦还反映出事业和人生的多样性。在事业的起步阶段，可能会面临混乱与危机亟待应对，只有通过正确的教育和坚毅的行动才能扭转局势。在商业上需谨慎行事，树立高尚的道德规范以及良好的信用，以增强竞争力。

总体而言，蒙卦告诫我们：在面对不确定性时应坚持学习与成长，切勿急于求成，同时在与人交往时，要关注他人的品德和真诚动机，方能在人生道路上通达顺利。


In [37]:
import csv
import datetime
import os

def main():
    # 确保 data 目录存在
    if not os.path.exists('data'):
        os.makedirs('data')

    # 解析 data/raw_data.txt 得到 raw_content_data 列表
    raw_content_data = []
    with open('data/raw_data.txt', 'r', encoding='utf-8') as file:
        content = file.read()
        data_samples = content.split('\n\n')
        for sample in data_samples:
            cleaned_sample = sample.strip()
            if cleaned_sample:
                raw_content_data.append(cleaned_sample)

    # 创建带有时间戳的CSV文件名
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"data/zhouyi_dataset_{timestamp}.csv"

    # 创建CSV文件并写入标题行
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(['content', 'summary'])

        # 循环遍历 raw_content_data 数据样例
        for raw_content in raw_content_data:
            # 调用 gen_data 方法得到 ai_message_content
            ai_message_content = gen_data(raw_content)

            # 解析 ai_message_content 得到 content 和 summary
            content, summary = dataset_parser(ai_message_content)
            
            print("Content:", content)
            print("Summary:", summary)

            # 调用 generate_question_summary_pairs 得到20组 pairs
            pairs = generate_question_summary_pairs(content, summary)

            # 将 pairs 写入 csv 文件
            for pair in pairs:
                writer.writerow(pair)


In [ ]:
# 执行主函数
main()

Content: "蒙卦
Summary: "蒙卦"
summary: "蒙卦在《周易》中象征启蒙与成长，它由上卦艮（山）和下卦坎（水）组成，展现了山下泉水喷涌而出的卦象，寓意在蒙昧状态中蕴含着生机与希望。此卦首爻占筮指示，强调不是我去求教于蒙昧之人，而是蒙昧之人自身带有求知的渴望。若持敬畏之心，初次占筮得以启示；反复轻慢则神灵不再感应，但整体仍是吉祥的象征。

哲学层面，蒙卦反映智慧未启发之前的懵懂状态与犹豫不决，提醒人们应顺应贤师良友的教导，以果敢坚毅的行动来培养德行。其象曰“山下出泉”，君子当效法如泉水般坚持不懈，促进自身品德成长。

蒙卦同时提示处于初创或迷惑状态时需忍耐和慎重，凡事多听取他人意见，方能运势通达。事业上虽有混乱与危机，但只要脚踏实地、戒除急功近利，定能转危为安。婚姻与社交须重视品德和宽容，决策时严谨果断，适时接受教诲和考验，最终可达通泰。

综观蒙卦，它不仅代表智慧的启蒙，也寓意着人生初期的磨砺与成长，鼓励以坚定信念和正确方法克服愚昧，开拓通达的未来。
Content: 屯卦
Summary: 屯卦是周易中的一个重要卦象，其卦名为“屯”，意指万物开始孕育、成长的艰难过程。由上卦坎（云）和下卦震（雷）构成，象征着云层中雷霆的震动。屯卦传达出一种生机与挑战并存的局面，强调在初始阶段往往面临困难，然只要坚守、努力，便可期见转机。

屯卦的核心解读为：大吉大利，但在实际行动上不宜急于进取。此卦提醒人们在处于困境时应保持冷静与稳重，付出努力之后才能迎来曙光。对于创业和事业而言，初期可能会遭遇诸多挫折，但如果能够坚持不懈、灵活应对，最终会有成功的可能。

传统的象辞进一步阐释道：“屯者难也”，即万事在起步阶段常常艰难，如同植物在生长初期面临风雨。然而，逆境中的坚持和智慧将引导人们走出困境。总之，屯卦强调在动中遇险，需以刚毅果敢的态度面对困难，以求在动荡中开创未来。
Content: {
    "content": "需卦",
    "summary": "在周易中，需卦是一个象征期待和耐心的卦象，由上卦坎（险）与下卦乾（健）相叠构成。此卦卦辞曰：“需。有孚，光亨，贞吉。利涉大川。”意味着亨通和吉利，尤其适合于渡水跨河之时。需卦象征着在移动与变动中需要等待合适的时机，强调在面对险阻时采用稳健的态度。云聚于天，暗示着未雨绸缪，以待时而动，君子由此出生机，

### 异常分析


训练第一个 epoch 时，Training Loss 比较奇怪：

```
Step	Training Loss
1	3.594100
2	4.049100
3	3.091200
4	3.381700
5	3.547800
6	2.610200
7	2.657900
8	3.163900
```

通过解析 gpt-4o-mini 生成结果发现问题

In [40]:
def gen_data(raw_content):
    """
    使用LangChain GPT-4o-mini调用处理单个数据样例。

    :param raw_content: 原始数据样例。
    :return: GPT-4o-mini模型生成的内容。
    """
    # 系统消息定义背景和任务
    system_message = SystemMessage(
        content="""
        你是中国古典哲学大师，尤其擅长周易的哲学解读。

        接下来，你收到的都是关于周易卦象的解释，你需要整理润色，并生成用于大模型训练的内容和格式。

        示例输入：

        师卦，此卦是异卦相叠，下卦为坎，上卦为坤。“师”指军队。坎为水、为险；坤为地、为顺，喻寓兵于农。兵凶战危，用兵乃圣人不得已而为之，但它可以顺利无阻碍地解决矛盾，因为顺乎形势，师出有名，故能化凶为吉。占得此卦，对于军事上率师出征非常有利，必无灾祸。师卦是天马出群之卦，以寡伏众之象。
        师卦位于讼卦之后，《序卦》之中这样解释道：“讼必有众起，故受之以师。师者，众也。”争讼的人越来越多，以致形成了军队。

        期待结果：

        content:"师卦"
        summary:"在周易中，师卦是一个极具深意的卦象，它由两个异卦相叠组成：下卦坎（水）和上卦坤（地）。这一卦象代表“师”，即军队，寓意着兵力和农力的结合。在这里，坎卦象征着水和险难，而坤卦象征着地和顺从，暗示着通过将军事力量安置于民间，可以在必要时顺利调动。

        师卦的核心哲学是：虽然兵力代表着危险和战争，但其使用应当是圣人不得已而为之的最后手段。在正确的情况下，军事力量可以顺应形势，将危险转化为吉祥。因此，在军事策略上，此卦象征着出征将会顺利，无灾祸。

        师卦紧随讼卦（争讼卦），在《序卦》中解释为“讼必有众起，故受之以师”。这意味着争端激化至众多人群的参与，形成了类似军队的集体力量。"

        返回格式要求：
        content:"{卦名}"
        summary:"{内容}"
        """
    )

    # 人类消息包含原始数据样例
    human_message = HumanMessage(
        content=raw_content
    )

    # 构建消息列表并进行模型调用
    messages = [system_message, human_message]
    ai_message = chat(messages)

    return ai_message.content

In [41]:
# 执行主函数
main()

Content: 蒙卦
Summary: 在周易中，蒙卦是一个富有启示意义的卦象，由上卦艮（山）和下卦坎（水）组成，象征着在山下涌出的泉水。此卦传达了‘蒙’即蒙昧的状态，强调智慧尚未开启、犹豫不决和缺乏果断的状态。因此，该卦常常暗示迷惑、疑虑与反复不定，初时可能面临困惑，但最终通过获得启发和教育能够通达亨利。

卦辞中提到：“不是我有求于幼稚愚昧的人，而是幼稚愚昧的人有求于我。”这说明，在更加成熟的状态中，君子应以果敢坚毅的行动去培养自身的德行。此卦象启示我们，光明的未来需要耐心、聆听他人的意见以及勇敢的行动来实现。

从不同角度分析，蒙卦在时运、事业、经商、求名、婚恋等方面均有其独特的指导意义。时运上，强调蓄积德行；事业上，尽管开始阶段可能混乱，但应通过教育与实践来转变；经商时，则需小心谨慎，注重商业道德；求名中，强调基础教育的重要性；婚恋则提醒关注对方的品德而非物质条件。蒙卦提醒我们，待机而动，审时度势，便能化解困境，迈向顺利之途。
Content: 屯卦
Summary: 屯卦象征着初生的困难与险境，它的构成由上下两个异卦组成：上卦为坎（云），下卦为震（雷）。这一卦在周易中被解释为大吉大利，意味着虽然初期艰难，但雷雨交加的环境中蕴藏着生机与希望。换言之，屯卦表达了万物萌发之初所必经历的挑战。

《象辞》强调，君子应借鉴云雷的特性，以云的恩泽和雷的威严来治理国家。这种治理需要谨慎与果敢，因屯卦本身指出，在危机和困难中宜守不宜进，需要施以耐心和努力，才能逐步排除困难，迎接成功。

从天机的角度来看，屯卦提醒我们身处困境时，务必小心翼翼，勇往直前，积极应对困难，寻求他人帮助，也必须在潜心积累后再求发展。对于事业和财运，初期创业可能会面临挫折，因此决策上应保持坚定信念，通过努力克服艰难。在婚恋方面，尽管可能经历波折，但始终坚持真诚的大步追求，能够实现美满的期望。

总之，屯卦教导我们在面临挑战时要有坚韧不拔的毅力和乐观的态度，才能在逆境中找到突破和成就的机会。
Content: 需卦
Summary: 需卦代表着等待与期待，其卦象由上卦坎（水）和下卦乾（天）组合而成，象征着云聚天上，预示着适时降雨。此卦的核心思想在于虽然时机尚未成熟，但若能耐心等待，最终会迎来好运。

需卦的本意在于应对险阻，其名虽然意含抓获俘虏，同时也体现了对时机的把握。古人有云：“云上于天，需；君